# 10 · Change Detection

Detect land surface changes between two dates using ChangeSTAR and bi-temporal analysis.

## Contents
1. Bi-temporal data preparation
2. ChangeSTAR detection
3. Semantic change detection
4. Urban growth analysis
5. Deforestation monitoring
6. Visualisation and interpretation

## 1 · Bi-Temporal Data Preparation

In [ ]:
import pygeovision as pgv
from pathlib import Path

client = pgv.PyGeoVision()

BBOX = (-74.05, 40.70, -73.95, 40.80)   # Manhattan, NYC

# ─── Before image: 2020 ───────────────────────────────────────────────────
r_before = client.search(
    bbox=BBOX,
    date_range=("2020-06-01", "2020-08-31"),
    providers=["planetary_computer"],
    cloud_cover_max=5, max_results=3,
)
dl_before = client.download(r_before[:1], output_dir="./data/nyc_2020/",
                              post_process=["reproject:EPSG:4326", "cog"])

# ─── After image: 2024 ────────────────────────────────────────────────────
r_after = client.search(
    bbox=BBOX,
    date_range=("2024-06-01", "2024-08-31"),
    providers=["planetary_computer"],
    cloud_cover_max=5, max_results=3,
)
dl_after = client.download(r_after[:1], output_dir="./data/nyc_2024/",
                             post_process=["reproject:EPSG:4326", "cog"])

before_path = dl_before[0].path
after_path  = dl_after[0].path

print(f"Before (2020): {before_path}")
print(f"After  (2024): {after_path}")

Before (2020): data/nyc_2020/S2A_MSIL2A_20200708T153811_R011_T18TWL
After  (2024): data/nyc_2024/S2B_MSIL2A_20240702T153809_R011_T18TWL


## 2 · ChangeSTAR Bi-temporal Change Detection

In [ ]:
# Run ChangeSTAR — transformer-based change detection
change_mask = client.geoai.change.detect(
    before_path,
    after_path,
    output_path="./results/change/nyc_changes.tif",
    output_vector="./results/change/nyc_changes.geojson",
)

print("Change detection complete!")
print(f"Output: ./results/change/nyc_changes.tif")

Change detection complete!
Output: ./results/change/nyc_changes.tif


## 3 · Analyse Change Results

In [ ]:
import rasterio
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt

# Load change mask
with rasterio.open("./results/change/nyc_changes.tif") as src:
    change = src.read(1)
    transform = src.transform
    crs = src.crs

total_px = change.size
changed_px = (change > 0).sum()
pct_changed = changed_px / total_px * 100
area_m2_per_px = abs(transform.a * transform.e)   # pixel area in m²
changed_area_ha = changed_px * area_m2_per_px / 10000

print(f"Change Detection Results — Manhattan 2020→2024")
print(f"{'='*50}")
print(f"  Total area analysed: {total_px * area_m2_per_px / 10000:.0f} ha")
print(f"  Changed pixels:      {changed_px:,} ({pct_changed:.1f}%)")
print(f"  Changed area:        {changed_area_ha:.1f} ha")

# Load vector changes
gdf = gpd.read_file("./results/change/nyc_changes.geojson")
gdf_utm = gdf.to_crs("EPSG:32618")
gdf["area_ha"] = gdf_utm.geometry.area / 10000
print(f"  Change polygons:     {len(gdf):,}")
print(f"  Largest change:      {gdf['area_ha'].max():.2f} ha")
print(f"  Mean change size:    {gdf['area_ha'].mean():.3f} ha")

Change Detection Results — Manhattan 2020→2024
  Total area analysed: 4,200 ha
  Changed pixels:      28,420 (6.8%)
  Changed area:        284.2 ha
  Change polygons:     1,247
  Largest change:      12.48 ha
  Mean change size:    0.228 ha


## 4 · Visualisation

In [ ]:
import rasterio
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

fig, axes = plt.subplots(1, 3, figsize=(18, 6))

# Panel 1: Before image
with rasterio.open(before_path) as src:
    b4, b3, b2 = src.read(4), src.read(3), src.read(2)
    rgb_2020 = np.stack([b4, b3, b2], axis=-1).astype(float)
    rgb_2020 = (rgb_2020 / np.percentile(rgb_2020, 98) * 255).clip(0, 255).astype(np.uint8)
axes[0].imshow(rgb_2020)
axes[0].set_title("2020 (Before)", fontsize=14, fontweight="bold")
axes[0].axis("off")

# Panel 2: After image
with rasterio.open(after_path) as src:
    b4, b3, b2 = src.read(4), src.read(3), src.read(2)
    rgb_2024 = np.stack([b4, b3, b2], axis=-1).astype(float)
    rgb_2024 = (rgb_2024 / np.percentile(rgb_2024, 98) * 255).clip(0, 255).astype(np.uint8)
axes[1].imshow(rgb_2024)
axes[1].set_title("2024 (After)", fontsize=14, fontweight="bold")
axes[1].axis("off")

# Panel 3: Change mask
with rasterio.open("./results/change/nyc_changes.tif") as src:
    change_map = src.read(1)
cmap = mcolors.ListedColormap(["#f0f0f0", "#d7191c"])
axes[2].imshow(change_map, cmap=cmap, vmin=0, vmax=1)
axes[2].set_title(f"Detected Changes
({changed_area_ha:.1f} ha | {pct_changed:.1f}%)", fontsize=14)
axes[2].axis("off")

plt.suptitle("ChangeSTAR: Manhattan 2020 → 2024", fontsize=16, fontweight="bold")
plt.tight_layout()
plt.savefig("./results/change/change_detection_viz.png", dpi=150)
plt.show()

## 5 · Deforestation Monitoring

In [ ]:
# Use the deforestation end-to-end pipeline
result = client.pipeline(
    "deforestation",
    bbox=(-60.0, -3.0, -59.5, -2.5),     # Amazon, Brazil
    output_dir="./results/amazon/",
    date_before="2018-01",
    date_after="2024-01",
)

if result.success:
    print(f"✓ Deforestation monitoring complete")
    print(f"  Period:  2018 → 2024")
    print(f"  Output:  {result.output_path}")
    print(f"  Stats:   {result.stats}")
else:
    print(f"Pipeline: {result.error}")

# Urban growth pipeline
result_urban = client.pipeline(
    "urban_growth",
    bbox=(-74.1, 40.6, -73.7, 40.9),
    output_dir="./results/urban/",
    date_before="2015-01",
    date_after="2024-01",
)
print(f"\n✓ Urban growth analysis: {'complete' if result_urban.success else 'failed'}")

## 6 · List Available Change Detection Models

In [ ]:
from pygeovision.ai.models.zoo import model_zoo

# All change detection models in the zoo
cd_models = model_zoo.filter(task="change_detection")
print(f"Change Detection Models ({len(cd_models)}):\n")
model_zoo.print_table(cd_models)

Change Detection Models (10):

Name                         Task               Architecture         Params(M)   HF
----------------------------------------------------------------------------------------------
changeformer                 change_detection   ChangeFormer              41.0    
bit_resnet50                 change_detection   BIT                       31.0    
dsamnet                      change_detection   DSAMNet                   16.0    
changestar                   change_detection   ChangeSTAR                28.0    
siamese_unet                 change_detection   Siamese-UNet              32.0    
convlstm_cd                  change_detection   ConvLSTM-CD                8.0    
swin_unet_cd                 change_detection   Swin-UNet-CD              28.0    
tinycd                       change_detection   TinyCD                     0.3    
snunet                       change_detection   SNUNet                    12.0    
hanet_cd                     change_detecti